In [ ]:
import os
os.environ["OPENAI_API_KEY"] ="sk-proj-"


In [2]:
%pip install -q langchain-openai langchain-core requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# TOOL CREATE

@tool
def multiply(a: int, b: int) ->int:
  """Given 2 numbers a and b this tool returns their product"""
  return a*b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))


12


In [6]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

### Tool Binding

In [7]:
llm = ChatOpenAI()
llm_with_tools=llm.bind_tools([multiply])

In [8]:
query = HumanMessage('can you multiply 3 with 1000')

In [9]:
messages = [query]

In [10]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [11]:
result = llm_with_tools.invoke(messages)
messages.append(result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 63, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E4ezDQWqeS3uvGsjdUX1Rkd8acWiw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8d27-0742-7c70-949b-b3ae48ce7cd9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_B1BvuqAxwab8MW1jF88Yq3uH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 18, 'total_tokens': 81, 'input_token

In [12]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 1000},
  'id': 'call_B1BvuqAxwab8MW1jF88Yq3uH',
  'type': 'tool_call'}]

In [13]:
tool_result = multiply.invoke(result.tool_calls[0])

In [14]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_B1BvuqAxwab8MW1jF88Yq3uH')

In [15]:
messages.append(tool_result)

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 63, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E4ezDQWqeS3uvGsjdUX1Rkd8acWiw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8d27-0742-7c70-949b-b3ae48ce7cd9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_B1BvuqAxwab8MW1jF88Yq3uH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 18, 'total_tokens': 81, 'input_token

In [17]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [18]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [19]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784764801,
 'time_last_update_utc': 'Thu, 23 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784851201,
 'time_next_update_utc': 'Fri, 24 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.6138}

In [20]:
convert.invoke({'base_currency_value':10, 'conversion_rate':96.3793})

963.793

In [21]:
llm = ChatOpenAI()

In [22]:
llm_with_new_tools = llm.bind_tools([convert,get_conversion_factor])

In [23]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [24]:
ai_message = llm_with_new_tools.invoke(messages)

In [25]:
messages.append(ai_message)

In [26]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_IVwzXXCpsgozj2JxwJfakhvM',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_vB5WJlGesaZxT8OZVrWcTgFJ',
  'type': 'tool_call'}]

In [27]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [28]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E4ezFtcZKE1H9Rpj5NPrSuOvtdFBT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8d27-15e4-7c62-8df9-09e2686dd711-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'call_IVwzXXCpsgozj2JxwJfakhvM', 'type': 'tool_call'

In [29]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR (Indian Rupee) and USD (United States Dollar) is 0.01035. Therefore, 10 INR is equivalent to approximately 0.1035 USD.'

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert],
    system_prompt="You are a helpful unit conversion assistant."
)


In [31]:
# --- Step 6: Run the Agent ---
user_query = 'What is the conversion factor between INR and EURO, and based on that can you convert 1 EURO to INR'

response = agent.invoke({"input": user_query})

In [32]:
response

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 102, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E4ezKO6sAuwsF8k3n45oBjH8ho7YW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f8d27-292f-7ff0-8627-2b4d4579f13d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'EUR'}, 'id': 'call_eLFe2d1ra2wmvjQ5npNtq2AK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 102, 'output_tokens': 21, 'total_tokens': 123, 'input_token_details': {'audio': 0, 'cache_read': 